# Paddle Phase 2 — Colab A100 (một GPU)
Chọn **Runtime → Change runtime type → A100 GPU**, bật High-RAM nếu có, rồi bấm nút ▶ từng cell từ trên xuống.
Clone mới phải chạy đủ Phase 1 (500 ảnh) và **bạn tự xem kết quả, điền review tại bước 4** trước khi chạy Phase 2.

Giữ detector của Triệu: PaddleOCR v3.3.3, Paddle GPU 3.3.1 cu126, checkpoint chính thức,
FP32/batch 1, resize 736/min, DBPostProcess 0.3/0.6/1.5. Không giảm ảnh, không đổi ngưỡng.
Tham chiếu ICDAR2015: P 58.522%, R 47.280%, Hmean 52.304%; không phải reproduction 83.8%.

Runtime được pin **f586e14ab8d63e5f63a12550ac447da4f04a79b2**: bản GPU-only đã có sửa lỗi pip,
trước thay đổi CPU retry. Vì CPU đã hết RAM ở phiên T4, bản này **không fallback CPU**.
Không pull HEAD giữa lượt chạy. A100 vẫn cần kiểm tra Phase 1; không mặc định số học giữa GPU hoàn toàn giống nhau.

Notebook tải 3 dataset từ Kaggle vào đĩa Colab. Nếu cần đăng nhập, thêm secret
`KAGGLE_API_TOKEN` và bật quyền notebook; không dán token vào cell.
TextOCR JSON không cần tải vì GT đã có trong manifest freeze.

Mount Google Drive để lưu bản sao tiến độ mỗi 15 phút và ZIP cuối cùng.
Bản sao tiến độ là **INCOMPLETE**, phục vụ giữ bằng chứng, không tự phục hồi gate ở clone mới.
Nếu runtime bị xóa, cần chạy lại từ đầu; nếu chỉ ngắt cell mà runtime/GPU vẫn giữ nguyên, có thể chạy lại bước 7.
Không trộn prediction T4 cũ vào lượt A100. Chưa xác nhận chạy hoàn tất trên A100.

## 0. Clone repo đúng commit, kiểm tra GPU

In [ ]:
import os, sys, re, json, time, signal, random, hashlib, shutil, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/ncat218/CS431.git'
REPO_BRANCH = 'paddle-phase2'   # nhánh chứa adapter tools/eval.py-equivalent (hoặc main sau khi merge)
REPO_REF = 'f586e14ab8d63e5f63a12550ac447da4f04a79b2'                   # tuỳ chọn: pin một commit SHA cụ thể; để trống = HEAD của REPO_BRANCH (SHA được ghi lại)
BASE = Path('/content')
assert BASE.is_dir(), 'Mở notebook trong Google Colab.'
INPUT_ROOT = BASE / 'cs431_data'
INPUT_ROOT.mkdir(exist_ok=True)
REPO = BASE / 'CS431_A100'
RUN_ROOT = REPO / 'outputs/paddle_colab_a100_v1'
PHASE1 = RUN_ROOT / 'phase1'
OUT2 = RUN_ROOT / 'phase2'
VENV = BASE / 'cs431-paddle-a100'
PYTHON = str(VENV / 'bin/python')
LOGS = BASE / 'cs431_logs'
LOGS.mkdir(parents=True, exist_ok=True)
CUDA_WHEEL = 'cu126'          # chỉ đổi sang cu118 TRƯỚC Phase 1 nếu driver không hỗ trợ CUDA 12.6
os.environ.pop('PYTHONPATH', None)


def run(cmd, log=None, cwd=None, env=None):
    """Chạy lệnh, in trực tiếp ra notebook và ghi log; lỗi thì dừng."""
    cmd = [str(c) for c in cmd]
    print('$', ' '.join(cmd), flush=True)
    fh = open(log, 'a', encoding='utf-8') if log else None
    try:
        with subprocess.Popen(cmd, cwd=cwd, env=env, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, text=True, bufsize=1) as p:
            for line in p.stdout:
                print(line, end='', flush=True)
                if fh:
                    fh.write(line)
    finally:
        if fh:
            fh.close()
    if p.returncode != 0:
        raise RuntimeError(f'Lệnh lỗi (exit {p.returncode}). Xem log: {log}')


if not (REPO / '.git').is_dir():
    run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, REPO])
elif not REPO_REF:
    branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=REPO, text=True).strip()
    assert branch == REPO_BRANCH, 'Clone thuộc nhánh khác; không tự đổi nhánh.'
    run(['git', 'pull', '--ff-only', 'origin', REPO_BRANCH], cwd=REPO)
if REPO_REF:
    current = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
    if current != REPO_REF:
        assert not (REPO / '.local').exists(), 'Clone đã có state khác; dùng thư mục clone mới.'
        run(['git', 'checkout', '--detach', REPO_REF], cwd=REPO)
HEAD = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
assert not REPO_REF or HEAD == REPO_REF, HEAD
dirty = subprocess.check_output(['git', 'status', '--porcelain', '--untracked-files=no'], cwd=REPO, text=True).strip()
assert not dirty, 'File tracked trong clone bị sửa:\n' + dirty
assert (REPO / 'cs431/ppocr_detector.py').is_file(), 'Clone chưa có adapter tools/eval.py-equivalent: sai nhánh/commit'
PIN = json.loads((REPO / 'paddle/configs/paddle_v5.json').read_text(encoding='utf-8'))
assert PIN['paddleocr_tag'] == 'v3.3.3' and PIN['package_versions']['paddlepaddle-gpu'] == '3.3.1', PIN
print('Repo:', REPO, '@', HEAD, '| PaddleOCR', PIN['paddleocr_tag'], '| paddle', PIN['package_versions']['paddlepaddle-gpu'])

gpus = subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total,driver_version',
                                '--format=csv,noheader'], text=True).strip().splitlines()
print(*gpus, sep='\n')
assert len(gpus) == 1 and 'A100' in gpus[0], 'Cần runtime một A100. Đổi accelerator rồi chạy lại.'
assert int(gpus[0].split(',')[2].strip().split()[0]) >= 38000, 'VRAM A100 không đủ/đang chia nhỏ.'
GPUS = '0'
print('RAM:', Path('/proc/meminfo').read_text().splitlines()[:3])
for name in ('memory.max', 'memory.peak'):
    p = Path('/sys/fs/cgroup') / name
    if p.exists(): print(name, p.read_text().strip())
print('Disk free GiB:', round(shutil.disk_usage(BASE).free / 2**30, 1))
print('GPU IDs cho workers:', GPUS)
subprocess.run([sys.executable, str(REPO / 'run.py'), 'paddle', 'status'], cwd=REPO)   # clone mới: BLOCKED là đúng

## 1. Môi trường riêng + model + Google Drive
Python 3.13 riêng do uv quản lý; kernel Colab không cần đổi phiên bản.
Không cài Paddle vào môi trường có Torch. Setup kiểm tra GPU, package pins và checkpoint SHA.
Drive yêu cầu bạn chọn tài khoản và cấp quyền qua giao diện Colab.

In [ ]:
t0 = time.time()
print('Colab kernel:', sys.version)
run([sys.executable, '-m', 'pip', 'install', 'uv'])
run([sys.executable, '-m', 'uv', 'python', 'install', '3.13'])
BOOTSTRAP_PYTHON = subprocess.check_output(
    [sys.executable, '-m', 'uv', 'python', 'find', '3.13'], text=True).strip()

run([BOOTSTRAP_PYTHON, REPO / 'paddle/tools/setup_paddle.py', '--venv', VENV, '--cuda', CUDA_WHEEL],
    LOGS / '02_setup_paddle.log', cwd=REPO)
run([PYTHON, REPO / 'paddle/tools/prepare_paddle_model.py'], LOGS / '03_prepare_model.log', cwd=REPO)
run([PYTHON, '-c', 'import importlib.metadata as m, json, sys; '
     'print(sys.version); print(json.dumps({k: m.version(k) for k in ["paddlepaddle-gpu","numpy","opencv-python","shapely","Pillow"]}))'],
    LOGS / '04_versions.log')
print(f'Môi trường xong sau {time.time() - t0:.0f} s')

from google.colab import drive
drive.mount('/content/drive')
RUN_LABEL = 'a100_' + time.strftime('%Y%m%d_%H%M%S')
DRIVE_OUT = Path('/content/drive/MyDrive/CS431_Paddle_Phase2') / RUN_LABEL
DRIVE_OUT.mkdir(parents=True, exist_ok=True)
print('Kết quả sẽ lưu tại:', DRIVE_OUT)

## 1b. Tải từng ảnh clean và degraded; không chờ ZIP
Kiểm tra một ảnh mỗi bộ trước, sau đó tải đủ 9.898 cặp theo manifest và kiểm SHA256 từng file. Chạy lại giữ ảnh đã đúng hash.
ICDAR dùng đường dẫn cache Kagglehub trả về. Thư mục tìm dữ liệu dùng liên kết file nên không tạo bản sao ảnh. Không tải về máy cá nhân.


In [ ]:
# Download individual clean AND degraded images; no TextVQA archive request.
import os, sys, json, hashlib, time, re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
run([sys.executable, '-m', 'pip', 'install', 'kagglehub'])
import kagglehub

DOWNLOAD_CACHE = BASE / 'cs431_data'
os.environ['KAGGLEHUB_CACHE'] = str(DOWNLOAD_CACHE)
# A separate view with file symlinks handles both normal and Colab-mounted caches.
# No image copies; following cells search only this unambiguous directory tree.
DATA_VIEW = BASE / 'cs431_inputs_verified'
DATA_VIEW.mkdir(parents=True, exist_ok=True)

def download_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 20), b''):
            h.update(block)
    return h.hexdigest()

def link_verified_file(source, target):
    source, target = Path(source).resolve(), Path(target)
    if target.is_file():
        if target.resolve() == source or download_sha(target) == download_sha(source):
            return
        raise RuntimeError('File có sẵn khác dữ liệu nguồn: ' + str(target))
    if target.is_symlink():
        target.unlink()  # Only a stale generated file link, never the cached image.
    target.parent.mkdir(parents=True, exist_ok=True)
    target.symlink_to(source)

run([PYTHON, REPO / 'run.py', 'shared', 'unpack'], cwd=REPO)
run([PYTHON, REPO / 'run.py', 'shared', 'verify'], cwd=REPO)
download_manifest = REPO / 'outputs/shared/shared_manifest_v1.jsonl'
assert download_sha(download_manifest) == '94e3a2ce989ce6a298d7ea30e7ce70f22ac676092b29df808bc3d33b426bf242'
with download_manifest.open(encoding='utf-8') as f:
    download_rows = [json.loads(line) for line in f if line.strip()]
assert len(download_rows) == 9898

downloads = {}
icdar_source = Path(kagglehub.dataset_download('bestofbests9/icdar2015'))
downloads['bestofbests9/icdar2015'] = str(icdar_source)
print('ICDAR cache thực tế:', icdar_source)
# Use the returned path even when Kagglehub mounts it under /kaggle/input.
for label, pattern in (('images', r'img_(\d+)\.(?:jpg|jpeg|png)'),
                       ('gt', r'gt_img_(\d+)\.txt')):
    hits = []
    for dp, _, names in os.walk(icdar_source):
        files = [name for name in names if re.fullmatch(pattern, name, re.I)]
        ids = {re.fullmatch(pattern, name, re.I).group(1) for name in files}
        if ids == {str(i) for i in range(1, 501)} and len(files) == 500:
            hits.append((Path(dp), files))
    if len(hits) != 1:
        raise RuntimeError(f'ICDAR {label}: cần đúng 1 thư mục test 500 file, thấy {len(hits)} trong {icdar_source}')
    folder, names = hits[0]
    for name in names:
        link_verified_file(folder / name, DATA_VIEW / 'icdar2015' / label / name)

DOWNLOAD_SPECS = {
    'clean': ('anhthu128/text-vqa-train/versions/1', 'textvqa/'),
    'degraded': ('anhthu128/textvqa-suy-thoi-10k-nh/versions/2', 'textvqa_realistic_v2_10k/'),
}
def download_image(condition, row):
    handle, prefix = DOWNLOAD_SPECS[condition]
    rel = row[condition + '_relpath']
    assert not Path(rel).is_absolute() and '..' not in Path(rel).parts
    expected = row[condition + '_sha256']
    target = DATA_VIEW / condition / rel
    if target.is_file() and download_sha(target) == expected:
        return str(target)
    remote_path = prefix + rel
    cached = DOWNLOAD_CACHE / 'datasets' / handle / remote_path
    if cached.is_file() and download_sha(cached) == expected:
        saved = cached
    else:
        for attempt in range(3):
            try:
                saved = Path(kagglehub.dataset_download(handle, path=remote_path))
                break
            except Exception as error:
                status = getattr(getattr(error, 'response', None), 'status_code', None)
                if status not in (429, 500, 502, 503, 504) or attempt == 2:
                    raise RuntimeError(f'Tải file thất bại: {handle} :: {remote_path}') from error
                time.sleep(5 * (attempt + 1))
    if download_sha(saved) != expected:
        raise RuntimeError('SHA256 khác manifest, không dùng ảnh này: ' + remote_path)
    link_verified_file(saved, target)
    return str(target)

# Prove both single-file routes work before starting the full downloads.
for condition in DOWNLOAD_SPECS:
    print('Kiểm tra tải riêng:', condition, flush=True)
    print('SINGLE FILE + HASH OK:', download_image(condition, download_rows[0]), flush=True)

for condition, (handle, prefix) in DOWNLOAD_SPECS.items():
    print(f'{condition}: tải 9.898 ảnh riêng; giữ file đã đúng SHA256.', flush=True)
    with ThreadPoolExecutor(max_workers=2) as pool:
        for done, saved in enumerate(pool.map(lambda row: download_image(condition, row), download_rows), 1):
            if done == 1 or done % 100 == 0 or done == 9898:
                print(f'{condition.upper()} VERIFIED {done}/9898', flush=True)
    downloads[handle] = dict(local_root=str(DATA_VIEW / condition), remote_prefix=prefix)

INPUT_ROOT = DATA_VIEW
(LOGS / 'dataset_downloads.json').write_text(json.dumps(downloads, indent=2))
print('Đủ 19.796 ảnh clean/degraded đã kiểm hash; ICDAR test đủ 500 ảnh + GT.')
# Probe exactly the image that exhausted T4 VRAM, in a short-lived GPU process.
# This is diagnostic only; predictions are not reused in the benchmark.
probe_images = list(INPUT_ROOT.rglob('21d60e0822064acb.jpg'))
assert len(probe_images) == 1, 'Không tìm thấy đúng ảnh TextVQA từng OOM; kiểm tra dữ liệu tải về.'
probe_code = """
import json, sys
from pathlib import Path
from cs431.ppocr_detector import OfficialDetector
import paddle
pin = json.loads(Path('paddle/configs/paddle_v5.json').read_text())
model = Path('paddle/models/PP-OCRv5_server_det')
det = OfficialDetector(model/'PaddleOCR', pin['config_relpath'], model/'weights'/pin['weights_file'])
polys, scores = det.predict(sys.argv[1])
paddle.device.cuda.synchronize()
print('A100_PROBE_OK', len(polys), 'boxes', 'resize=', det.resize)
"""
probe_env = os.environ.copy()
probe_env['CUDA_VISIBLE_DEVICES'] = '0'
run([PYTHON, '-c', probe_code, probe_images[0]], LOGS / '05_a100_probe.log', cwd=REPO, env=probe_env)

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

def stream_sha(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

def snapshot(status):
    # Incomplete snapshots may contain a partial last JSONL line; never label them completed.
    try:
        bundle = BASE / f'CS431_A100_{status}.zip'
        with ZipFile(bundle, 'w', compression=ZIP_DEFLATED, compresslevel=1) as z:
            z.writestr('STATUS.json', json.dumps(dict(status=status, repo_commit=HEAD, gpu=GPUS,
                saved_at=time.strftime('%Y-%m-%dT%H:%M:%S'), automatic_resume=False)))
            for folder, prefix in ((PHASE1, 'phase1'), (OUT2, 'phase2'), (LOGS, 'logs')):
                for p in sorted(folder.rglob('*')):
                    if p.is_file() and not p.name.endswith('.lock'):
                        z.write(p, f'{prefix}/{p.relative_to(folder)}')
            comparison = REPO / 'comparison/phase2.md'
            if status == 'COMPLETE' and comparison.is_file():
                z.write(comparison, 'comparison/phase2.md')
        digest = stream_sha(bundle)
        dest = DRIVE_OUT / bundle.name
        temp = dest.with_suffix('.zip.uploading')
        shutil.copyfile(bundle, temp)
        assert stream_sha(temp) == digest, 'Drive copy checksum mismatch'
        temp.replace(dest)
        dest.with_suffix('.zip.sha256').write_text(digest + '\n')
        print('Đã lưu Drive:', dest, '| SHA256:', digest, flush=True)
    except Exception as e:
        print('LƯU DRIVE THẤT BẠI:', type(e).__name__, str(e), flush=True)
        if status == 'COMPLETE': raise

## 2. Phase 1 gate — ICDAR2015 test đủ 500 ảnh
Tự tìm thư mục có **đúng** `img_1 … img_500` và `gt_img_1.txt … gt_img_500.txt` (tập train có 1.000 ảnh nên không khớp).
Có nhiều ứng viên thì dừng và điền tay `ICDAR_IMAGES` / `ICDAR_GT`. Không tải mirror khác.

In [ ]:
ICDAR_IMAGES = ''   # để trống = tự tìm
ICDAR_GT = ''
DATASET_SOURCE = '' # để trống = tự ghi theo dataset bestofbests9/icdar2015 nếu đường dẫn khớp


def dirs_with_exact_ids(root, pattern, n=500):
    want = {str(i) for i in range(1, n + 1)}
    hits = []
    for dp, dn, fn in os.walk(root):
        ids = {m.group(1) for f in fn if (m := re.fullmatch(pattern, f, re.I))}
        if ids == want:
            hits.append(Path(dp))
    return hits


if not ICDAR_IMAGES or not ICDAR_GT:
    img_hits = dirs_with_exact_ids(INPUT_ROOT, r'img_(\d+)\.(?:jpg|jpeg|png)')
    gt_hits = dirs_with_exact_ids(INPUT_ROOT, r'gt_img_(\d+)\.txt')
    print('Ứng viên ảnh:', img_hits)
    print('Ứng viên GT :', gt_hits)
    if len(img_hits) != 1 or len(gt_hits) != 1:
        raise RuntimeError('Cần đúng 1 thư mục ảnh test và 1 thư mục GT test; điền tay ICDAR_IMAGES/ICDAR_GT.')
    ICDAR_IMAGES = ICDAR_IMAGES or str(img_hits[0])
    ICDAR_GT = ICDAR_GT or str(gt_hits[0])
if not DATASET_SOURCE:
    if 'bestofbests9' in ICDAR_IMAGES or 'icdar2015' in ICDAR_IMAGES.lower():
        DATASET_SOURCE = (f'Kaggle https://www.kaggle.com/datasets/bestofbests9/icdar2015 ; images={ICDAR_IMAGES} ; '
                          f'gt={ICDAR_GT} ; cùng nguồn ảnh với paddle/phase1/report/official_eval')
    else:
        raise ValueError('Không nhận ra nguồn ICDAR2015; điền DATASET_SOURCE thật.')
print('ICDAR_IMAGES =', ICDAR_IMAGES)
print('ICDAR_GT     =', ICDAR_GT)
print('SOURCE       =', DATASET_SOURCE)

t0 = time.time()
run([PYTHON, REPO / 'run.py', 'paddle', 'phase1', '--images', ICDAR_IMAGES, '--gt', ICDAR_GT,
     '--dataset-source', DATASET_SOURCE, '--gpus', GPUS, '--out', PHASE1], LOGS / '10_phase1.log', cwd=REPO)
print(f'Phase 1 xong sau {time.time() - t0:.0f} s')

## 3. Xem kết quả Phase 1 trước khi quyết định
Xem P/R/Hmean, counts, log workers, `detector.json` (553/553 tham số, 0 thiếu, 0 lệch shape; resize 736/min) và **20 overlays**
(xanh lá = GT valid, xám = GT ignored `###`, đỏ = prediction).

**Kiểm tra tái tạo:** cùng code/config/weights/packages với `tools/eval.py` của Triệu (P 58,52 / R 47,28 / Hmean 52,30) nên số phải
gần như trùng. Nếu lệch, kiểm tra dữ liệu/runtime/adapter trước khi review; chưa có ngưỡng sai số GPU được xác lập.

83,8 vẫn NOT VERIFIED. Người thực hiện tự quyết định sau khi xem bằng chứng.

In [ ]:
from IPython.display import display, Markdown
from PIL import Image

display(Markdown((PHASE1 / 'REVIEW_ME.md').read_text(encoding='utf-8')))
m1 = json.loads((PHASE1 / 'metrics.json').read_text())
print(json.dumps(m1, indent=2)[:3000])
per = [json.loads(l) for l in (PHASE1 / 'per_image.jsonl').read_text().splitlines() if l.strip()]
print('Ảnh không có prediction nào:', sum(r['predictions'] == 0 for r in per), '/ 500')
REF = PIN['phase1_official_reference']
for k in ('precision', 'recall', 'hmean'):
    print(f'{k:9s} repo {100 * m1["measured"][k]:.3f}%  | tools/eval.py {100 * REF[k]:.3f}%  | Δ {100 * (m1["measured"][k] - REF[k]):+.3f} pp')
print(json.dumps(json.loads(next((PHASE1 / 'inference').glob('gpu*/detector.json')).read_text()), indent=2))
for log in sorted((PHASE1 / 'inference').glob('gpu*.log')):
    print('---', log.name); print('\n'.join(log.read_text(errors='ignore').splitlines()[-5:]))
print(json.dumps(json.loads(next((PHASE1 / 'inference').glob('gpu*/runtime.json')).read_text()), indent=2))
for image in sorted((PHASE1 / 'overlays').glob('*.jpg')):
    print(image.name)
    with Image.open(image) as src:
        preview = src.copy(); preview.thumbnail((1000, 700)); display(preview)

## 4. Người thực hiện ghi nhận đánh giá Phase 1 (Trí tự điền)
Chỉ điền **sau khi đã xem** cell trên. Ghi P/R/Hmean thực tế và chênh lệch so với `tools/eval.py` 52,30, nhận xét overlays
(box theo dòng vs GT theo từ, bỏ sót chữ dọc/xoay…), nguồn dữ liệu và giới hạn (83,8 NOT VERIFIED).
Cell mặc định dừng — đây là quyết định của người thực hiện, không phải bước tự approve.

In [ ]:
REVIEWER = ''
REVIEW_NOTES = ''
if not REVIEWER.strip() or len(REVIEW_NOTES.strip()) < 20:
    raise ValueError('Xem kết quả Phase 1 rồi điền REVIEWER và REVIEW_NOTES (nhận định thực tế) để mở gate Phase 2.')
run([PYTHON, REPO / 'run.py', 'paddle', 'review', '--run', PHASE1, '--reviewer', REVIEWER, '--notes', REVIEW_NOTES],
    LOGS / '11_review.log', cwd=REPO)
run([PYTHON, REPO / 'run.py', 'paddle', 'status'], cwd=REPO)

## 5. Đọc nguồn shared, giải nén manifest đã freeze
Bắt buộc theo AGENTS.md: notebook shared, protocol, data audit, `docs/PHASE2_SHARED_MANIFEST.md`.
Manifest giải nén phải có SHA256 `94e3a2ce…f242` (9.898 cặp). Không join/sample lại.

In [ ]:
public = json.loads((REPO / 'shared/Shared_Manifest_TextVQA_TextOCR_Public.ipynb').read_text())
for cell in public['cells']:
    if cell['cell_type'] == 'markdown':
        display(Markdown(''.join(cell['source'])))
display(Markdown((REPO / 'docs/PHASE2_SHARED_MANIFEST.md').read_text()))
display(Markdown((REPO / 'shared/frozen/data_audit.md').read_text()))
print((REPO / 'shared/frozen/evaluation_protocol.json').read_text())
run([PYTHON, REPO / 'run.py', 'shared', 'unpack'], cwd=REPO)
run([PYTHON, REPO / 'run.py', 'shared', 'verify'], cwd=REPO)
MANIFEST = REPO / 'outputs/shared/shared_manifest_v1.jsonl'
rows = [json.loads(l) for l in MANIFEST.open(encoding='utf-8') if l.strip()]
assert len(rows) == 9898, len(rows)
print('Manifest rows:', len(rows), '| ví dụ:', rows[0]['clean_relpath'], '|', rows[0]['degraded_relpath'])

## 6. Xác định `clean_root` / `degraded_root` từ file thật
Root đúng là thư mục sao cho `root / clean_relpath` (vd `train_images/<id>.jpg`) và `root / degraded_relpath`
(vd `main/images/realistic_mix/L2/<id>.png`) tồn tại. Cell kiểm **toàn bộ 9.898 + 9.898 file** tồn tại và
SHA256 của 50 cặp ngẫu nhiên khớp manifest (workers còn hash lại từng ảnh trước inference).

In [ ]:
CLEAN_ROOT = ''      # để trống = tự tìm
DEGRADED_ROOT = ''


def find_root(relpaths, label, probe=20):
    sample = relpaths[:probe]
    hits = []
    for dp, dn, fn in os.walk(INPUT_ROOT):
        d = Path(dp)
        if all((d / r).is_file() for r in sample[:2]) and all((d / r).is_file() for r in sample):
            hits.append(d)
        if len(d.relative_to(INPUT_ROOT).parts) >= 10:
            dn[:] = []
    print(label, 'ứng viên:', hits)
    if len(hits) != 1:
        raise RuntimeError(f'{label}: cần đúng 1 root, thấy {len(hits)}; điền tay.')
    return str(hits[0])


CLEAN_ROOT = CLEAN_ROOT or find_root([r['clean_relpath'] for r in rows], 'clean_root')
DEGRADED_ROOT = DEGRADED_ROOT or find_root([r['degraded_relpath'] for r in rows], 'degraded_root')
missing = [r['source_image_id'] for r in rows if not (Path(CLEAN_ROOT) / r['clean_relpath']).is_file()
           or not (Path(DEGRADED_ROOT) / r['degraded_relpath']).is_file()]
assert not missing, f'{len(missing)} source thiếu file, ví dụ {missing[:5]}'


def file_sha(p):
    h = hashlib.sha256()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()


for r in random.Random(0).sample(rows, 50):
    assert file_sha(Path(CLEAN_ROOT) / r['clean_relpath']) == r['clean_sha256'], r['source_image_id']
    assert file_sha(Path(DEGRADED_ROOT) / r['degraded_relpath']) == r['degraded_sha256'], r['variant_id']
LOCATIONS = REPO / 'outputs/dataset_locations.json'
LOCATIONS.parent.mkdir(parents=True, exist_ok=True)
LOCATIONS.write_text(json.dumps(dict(clean_root=CLEAN_ROOT, degraded_root=DEGRADED_ROOT), indent=2))
print('OK: đủ 19.796 file, 50 cặp hash khớp. Locations:', LOCATIONS.read_text())

## 7. Phase 2 — một worker GPU 0, đủ 19.796 ảnh
Giữ đúng manifest/evaluator; chấm điểm và bootstrap 1.000 lần sau khi inference hoàn tất.
Cell in tiến độ mỗi phút; lưu bản sao INCOMPLETE vào Drive mỗi 15 phút và khi dừng/lỗi.
Nếu OOM: dừng và gửi log; không giảm resize hoặc chạy CPU.
Chỉ chạy lại cell này để resume khi cùng clone/runtime/GPU/output; không chạy đồng thời hai controller.

In [ ]:
cmd = [PYTHON, str(REPO / 'run.py'), 'paddle', 'phase2', '--locations', str(LOCATIONS), '--gpus', GPUS, '--out', str(OUT2)]
print('$', ' '.join(cmd))
t0 = time.time()
last_backup = time.time()
log_path = LOGS / '20_phase2.log'
with open(log_path, 'a', encoding='utf-8') as log:
    proc = subprocess.Popen(cmd, cwd=REPO, stdout=log, stderr=subprocess.STDOUT)
    try:
        while proc.poll() is None:
            time.sleep(60)
            parts = []
            for g in sorted((OUT2 / 'inference').glob('gpu*.log')):
                prog = [l for l in g.read_text(errors='ignore').splitlines() if l.startswith('PROGRESS')]
                parts.append(f'{g.stem}: {prog[-1] if prog else "khởi động"}')
            if time.time() - last_backup >= 900:
                snapshot('INCOMPLETE')
                last_backup = time.time()
            last = log_path.read_text(errors='ignore').splitlines()[-1:] or ['']
            print(f'[{(time.time() - t0) / 60:6.1f} phút]', ' | '.join(parts), '|', last[0][:120], flush=True)
    except KeyboardInterrupt:
        proc.send_signal(signal.SIGINT)   # controller tự dừng workers và gỡ lock
        proc.wait()
        raise
    finally:
        snapshot('INCOMPLETE')
print('\n'.join(log_path.read_text(errors='ignore').splitlines()[-30:]))
if proc.returncode != 0:
    raise RuntimeError(f'Phase 2 lỗi (exit {proc.returncode}); xem {log_path} và {OUT2}/inference/gpu*.log')
print(f'Phase 2 xong sau {(time.time() - t0) / 60:.1f} phút')
display(Markdown((OUT2 / 'metrics/clean_vs_degraded.md').read_text()))

## 8. Bảng so sánh với Monkey (cùng manifest/protocol/evaluator) + phân tích phụ
`run.py compare` từ chối nếu khác hash hoặc khác số source/variant. Latency khác phần cứng → không xếp hạng tốc độ.

In [ ]:
run([PYTHON, REPO / 'run.py', 'compare', '--paddle-metrics', OUT2 / 'metrics/metrics.json'],
    LOGS / '30_compare.log', cwd=REPO)
display(Markdown((REPO / 'comparison/phase2.md').read_text()))

pm = json.loads((OUT2 / 'metrics/metrics.json').read_text())
mm = json.loads((REPO / 'monkey/phase2/report/metrics.json').read_text())
lines = ['| Model | Cond | <16px | 16–32px | ≥32px |', '|---|---|---:|---:|---:|']
for name, m in (('paddle', pm), ('monkey', mm)):
    for cond in ('clean', 'degraded'):
        b = m['size_bins'][cond]
        lines.append(f'| {name} | {cond} | ' + ' | '.join(f'{100 * b[k]["recall"]:.2f}' for k in
                     ('small_lt16px', 'medium_16_32px', 'large_ge32px')) + ' |')
display(Markdown('**Recall theo chiều cao bbox GT gốc (%)**\n\n' + '\n'.join(lines)))
lines = ['| Model | Stratum | Hmean clean | Hmean degraded | Δ (pp) |', '|---|---|---:|---:|---:|']
for name, m in (('paddle', pm), ('monkey', mm)):
    for key, s in m['strata'].items():
        lines.append(f'| {name} | {key} | {100 * s["clean_paired"]["hmean"]:.2f} | '
                     f'{100 * s["degraded"]["hmean"]:.2f} | {s["delta_pp"]["hmean"]:+.2f} |')
display(Markdown('**Theo có/không perspective**\n\n' + '\n'.join(lines)))
counts = {c: [0, 0, 0] for c in ('clean', 'degraded')}
with (OUT2 / 'inference/predictions.jsonl').open() as f:
    for line in f:
        if not line.strip(): continue
        p = json.loads(line); count = counts[p['condition']]
        count[0] += 1; count[1] += int(not p['polygons']); count[2] += len(p['polygons'])
for cond, (total, empty, boxes) in counts.items():
    print(cond, '| ảnh rỗng:', empty, '/', total, '| tổng polygons:', boxes)
print('Gallery overlays:', OUT2 / 'overlays')
print('Thiết bị / thời gian:', (OUT2 / 'inference/timing.json').read_text())

## 9. Lưu kết quả hoàn tất vào Google Drive
ZIP gồm report Phase 1, Phase 2, logs, bảng so sánh và thông tin phiên bản.
Chỉ gắn COMPLETE khi đủ 19.796 prediction và metrics đã sinh. Không commit ZIP/pixels lên git.

In [ ]:
assert (OUT2 / 'metrics/metrics.json').is_file(), 'Phase 2 chưa chấm xong.'
with (OUT2 / 'inference/predictions.jsonl').open() as f:
    assert sum(bool(line.strip()) for line in f) == 19796
(LOGS / 'pip_freeze_venv.txt').write_text(subprocess.check_output([PYTHON, '-m', 'pip', 'freeze'], text=True))
(LOGS / 'repo_head.txt').write_text(HEAD + '\n')
snapshot('COMPLETE')
print('Xong. Mở Google Drive → MyDrive → CS431_Paddle_Phase2 →', RUN_LABEL)